In [1]:
import os
import pandas as pd

from metamers.behavioral.metadata import load_metadata
from metamers.behavioral.loader import get_valid_files
from metamers.behavioral.parser import parse_behavior_file
from metamers.behavioral.table import build_behavior_table

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------
BASE = "/mnt/c/Users/zevaz/GUI_beh_metamers/BehData"
CSV = f"{BASE}/participant_beh_recordPY.csv"

OUTPUT_DIR = "/home/zevaz/projects/metamers/behavioral"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TABLE_PATH = os.path.join(OUTPUT_DIR, "behavioral_table.csv")
PARQUET_PATH = os.path.join(OUTPUT_DIR, "behavioral_table.parquet")

# ---------------------------------------------------------
# 2. Load metadata
# ---------------------------------------------------------
df_full, df_to_process = load_metadata(CSV)

# If nothing to process, exit early
if df_to_process.empty:
    print("No new behavioral data to add.")
    exit(0)

# ---------------------------------------------------------
# 3. Build new behavioral rows
# ---------------------------------------------------------
df_new = build_behavior_table(
    metadata_df=df_to_process,
    base_path=BASE,
    loader=get_valid_files,
    parser=parse_behavior_file
)

# ---------------------------------------------------------
# 4. Load existing table (if any)
# ---------------------------------------------------------
if os.path.exists(TABLE_PATH):
    df_existing = pd.read_csv(TABLE_PATH)
else:
    df_existing = pd.DataFrame()

# ---------------------------------------------------------
# 5. Append new rows
# ---------------------------------------------------------
df_final = pd.concat([df_existing, df_new], ignore_index=True)

# ---------------------------------------------------------
# 6. Save updated table
# ---------------------------------------------------------
df_final.to_csv(TABLE_PATH, index=False)
df_final.to_parquet(PARQUET_PATH, index=False)

# ---------------------------------------------------------
# 7. Mark processed rows in metadata
# ---------------------------------------------------------
df_full.loc[df_to_process.index, "added"] = 1
df_full.to_csv(CSV, index=False)

print(f"Added {len(df_new)} new rows. Behavioral table updated.")


Added 20 new rows. Behavioral table updated.


In [ ]:
# from metamers.behavioral.metadata import load_metadata
# from metamers.behavioral.loader import get_valid_files
# from metamers.behavioral.parser import parse_behavior_file
# from metamers.behavioral.table import build_behavior_table
# import os

# BASE = "/mnt/c/Users/zevaz/GUI_beh_metamers/BehData"
# CSV = f"{BASE}/participant_beh_recordPY.csv"

# # Load metadata
# df_full, df_to_process = load_metadata(CSV)

# # Build behavioral table
# df_behavior = build_behavior_table(
#     metadata_df=df_to_process,
#     base_path=BASE,
#     loader=get_valid_files,
#     parser=parse_behavior_file
# )
# # uv guarantees: os.getcwd() == project root
# OUTPUT_DIR = "/home/zevaz/projects/metamers/behavioral"
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# df_behavior.to_csv(f"{OUTPUT_DIR}/behavioral_table.csv", index=False)
# df_behavior.to_parquet(f"{OUTPUT_DIR}/behavioral_table.parquet", index=False)

# # Mark processed rows
# df_full.loc[df_to_process.index, "added"] = 1
# df_full.to_csv(CSV, index=False)
